In [ ]:
!pip install xlrd -q

import pandas as pd, re
from google.colab import files

# ── 1. Upload: 3 WoS .xls exports (base corpus) + 1 Scopus .csv ──
uploaded = files.upload()
wos_files = [f for f in uploaded if f.endswith('.xls')]
sc_file   = [f for f in uploaded if f.endswith('.csv')][0]

wos = pd.concat([pd.read_excel(f) for f in wos_files], ignore_index=True)
sc  = pd.read_csv(sc_file)
print("WoS records:", len(wos), "| unique IDs:", wos['UT (Unique WOS ID)'].nunique())
print("Scopus records:", len(sc))

# ── 2. Normalise DOIs and titles for matching ──
def norm_doi(x):
    if pd.isna(x): return None
    return re.sub(r'^https?://(dx\.)?doi\.org/', '', str(x).strip().lower()) or None

def norm_txt(t):
    if pd.isna(t): return None
    return re.sub(r'[^a-z0-9]', '', str(t).lower()) or None

wos['doi'] = wos['DOI'].map(norm_doi);            sc['doi'] = sc['DOI'].map(norm_doi)
wos['nt']  = wos['Article Title'].map(norm_txt);  sc['nt']  = sc['Title'].map(norm_txt)

# ── 3. Match: DOI first, exact-title fallback ──
wos_dois, wos_titles = set(wos.doi.dropna()), set(wos.nt.dropna())
sc_dois,  sc_titles  = set(sc.doi.dropna()),  set(sc.nt.dropna())
sc['in_wos']  = sc.doi.isin(wos_dois) | sc.nt.isin(wos_titles)
wos['in_sc']  = wos.doi.isin(sc_dois) | wos.nt.isin(sc_titles)

overlap = sc.in_wos.sum()
union   = len(wos) + len(sc) - overlap
print(f"\nOverlap: {overlap}")
print(f"  share of Scopus corpus: {overlap/len(sc):.1%}")
print(f"  share of WoS corpus:    {wos.in_sc.mean():.1%}")
print(f"  WoS / (WoS ∪ Scopus):   {len(wos)/union:.1%}")
print(f"  matched via DOI only:   {sc.doi.isin(wos_dois).sum()}")

# ── 4. Profile the WoS-only records (what the Scopus subject filter loses) ──
wos_only = wos[~wos.in_sc]
print(f"\nWoS-only: {len(wos_only)} — top journals:")
print(wos_only['Source Title'].value_counts().head(10).to_string())

# ── 5. Profile the Scopus-only records ──
sc_only = sc[~sc.in_wos].copy()
print(f"\nScopus-only: {len(sc_only)} — top journals:")
print(sc_only['Source title'].value_counts().head(10).to_string())

# journal-level vs record-level misses
wos_journals = set(wos['Source Title'].map(norm_txt).dropna())
in_corpus_j  = sc_only['Source title'].map(norm_txt).isin(wos_journals)
print(f"\nScopus-only from journals ABSENT from WoS corpus: "
      f"{(~in_corpus_j).sum()} ({(~in_corpus_j).mean():.0%})")

# where does the search term actually appear?
pat = re.compile(r'life[\s\-]?expectanc', re.I)
def term_location(r):
    if pat.search(str(r['Title'])):           return 'title'
    if pat.search(str(r['Abstract'])):        return 'abstract'
    if pat.search(str(r['Author Keywords'])): return 'author_kw'
    if pat.search(str(r['Index Keywords'])):  return 'INDEX_KW_ONLY'
    return 'nowhere'
sc_only['term_loc'] = sc_only.apply(term_location, axis=1)
print("\nScopus-only — where 'life expectanc*' appears:")
print(sc_only.term_loc.value_counts().to_string())

# citation impact comparison
print(f"\nMedian citations — overlap: {sc[sc.in_wos]['Cited by'].median()}"
      f" | Scopus-only: {sc_only['Cited by'].median()}")

# ── 6. Export the Scopus-only list ──
sc_only[['Title','Source title','Year','DOI','Cited by','term_loc']] \
    .to_csv('scopus_only_records.csv', index=False)
files.download('scopus_only_records.csv')

Saving savedrecs 3.xls to savedrecs 3.xls
Saving savedrecs.xls to savedrecs.xls
Saving savedrecs2.xls to savedrecs2.xls
Saving scopus_export_Aug 4-2026_817d4861-5fa3-4f5a-b9b1-07b33c6a8991.csv to scopus_export_Aug 4-2026_817d4861-5fa3-4f5a-b9b1-07b33c6a8991.csv
WoS records: 2580 | unique IDs: 2580
Scopus records: 2712

Overlap: 1417
  share of Scopus corpus: 52.2%
  share of WoS corpus:    54.8%
  WoS / (WoS ∪ Scopus):   66.6%
  matched via DOI only:   1343

WoS-only: 1165 — top journals:
Source Title
PHARMACOECONOMICS                                    98
VALUE IN HEALTH                                      90
HEALTH ECONOMICS                                     67
JOURNAL OF HEALTH ECONOMICS                          54
JOURNAL OF MEDICAL ECONOMICS                         45
ECONOMICS & HUMAN BIOLOGY                            18
ECONOMIC AND SOCIAL CHANGES-FACTS TRENDS FORECAST    18
REVIEW OF DEVELOPMENT ECONOMICS                      12
HEALTH ECONOMICS REVIEW                      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>